In [2]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

I den här lektionen tittar vi på **hur vi skriver snabbare Pandas‑kod**.

Mål med lektionen:

- Förstå *när* vi behöver bry oss om prestanda.
- Se varför `for`‑loopar och `df.apply(axis=1)` ofta är långsamma.
- Lära oss skriva samma logik med **vektoriserade operationer**.
- Lära oss mäta prestanda på ett enkelt sätt i Jupyter.
- Få en första känsla för **minne och datatyper (dtypes)** i Pandas.


### Datasetet

Vi laddar in ett enkelt dataset innehållande energidata, med en kolumn `date_time` och en kolumn `energy_kwh`.

Låt oss läsa in det i en DataFrame. 

In [3]:
demand_profile_df = pd.read_csv('../data/demand_profile.csv')

demand_profile_df.head()

,date_time,energy_kwh
0,1/1/13 0:00,0.586
1,1/1/13 1:00,0.580
2,1/1/13 2:00,0.572
3,1/1/13 3:00,0.596
4,1/1/13 4:00,0.592


In [7]:
demand_profile_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8760 entries, 0 to 8759
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   date_time   8760 non-null   object 
 1   energy_kwh  8760 non-null   float64
dtypes: float64(1), object(1)
memory usage: 137.0+ KB


---
## 1. Mäta prestanda i Jupyter

Det finns två vanliga sätt vi kommer använda:
- **`%timeit`** (cell‑magisk) – kör koden flera gånger och tar medelvärdet.
- **`time.time()`** – bra för längre körningar / hela funktioner.


In [127]:
# time.time()

start = time.time()

demand_profile_df['energy_kwh'] * 1.05

end = time.time()

print(f'The time taken was: {end-start} seconds.')

The time taken was: 0.0002684593200683594 seconds.


In [122]:
# %timeit

%timeit -r 1 -n 10 demand_profile_df['energy_kwh'] * 1.05

68.8 μs ± 0 ns per loop (mean ± std. dev. of 1 run, 10 loops each)


---
## 2. Datum/tid – snabbare `to_datetime`

Vår kolumn `date_time` är just nu troligen `object` (strängar). Vi vill göra om den till `datetime`.

### 2.1 Utan format (långsammare)
Pandas måste gissa formatet på datumet.


In [141]:
%timeit -r 1 -n 10 pd.to_datetime(demand_profile_df['date_time'])

<magic-timeit>:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
<magic-timeit>:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
<magic-timeit>:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
<magic-timeit>:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
<magic-timeit>:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify 

206 ms ± 0 ns per loop (mean ± std. dev. of 1 run, 10 loops each)


<magic-timeit>:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.


### 2.2 Med angivet format (snabbare)

Vi talar om för Pandas hur strängen ser ut, t.ex. `'%d/%m/%y %H:%M'`.

> **Övning:** Kör cellerna och jämför tiderna.


In [ ]:
%timeit -r 1 -n 10 pd.to_datetime(demand_profile_df['date_time'] , format='%d/%m/%y %H:%M')

10 ms ± 0 ns per loop (mean ± std. dev. of 1 run, 10 loops each)


### 2.3 Spara den snabbare varianten i DataFrame

Nu gör vi själva omvandlingen en gång och sparar resultatet i en kolumn.


In [ ]:
demand_profile_df['date_time'] = pd.to_datetime(demand_profile_df['date_time'], format='%d/%m/%y %H:%M')

demand_profile_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8760 entries, 0 to 8759
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   date_time   8760 non-null   datetime64[ns]
 1   energy_kwh  8760 non-null   float64       
dtypes: datetime64[ns](1), float64(1)
memory usage: 137.0 KB


---
## 3. Rad‑vis logik vs vektoriserade operationer

Nu ska vi se hur olika sätt att beräkna samma sak kan skilja sig **massor** i prestanda.

Vi skapar en DataFrame med många rader:


In [143]:
# Skapa ett lite större exempel-dataset
n = 500000  # antal rader (kan höjas)

df = pd.DataFrame({
    "A": np.random.randint(0, 100, n),
    "B": np.random.randint(0, 100, n),
    "C": np.random.randint(0, 100, n),
})

df.head()

,A,B,C
0,65,29,34
1,72,60,52
2,63,97,24
3,95,47,27
4,37,96,23


Vi vill beräkna en ny kolumn `Sum = A + B + C` på fyra olika sätt:

1. **Dålig**: vanlig `for`‑loop + `df.loc[...]`
2. **Fortfarande dålig**: `df.iterrows()`
3. **Mindre dålig**: `df.apply(function, axis=1)`
4. **Bra**: vektoriserat uttryck `df["A"] + df["B"] + df["C"]`

Vi kommer mäta tiden för varje metod.


**Manual for-loop**

In [157]:
for_loop_df = df.copy()

start = time.time()

my_sums = []

for index in df.index:
    my_sums.append(df.loc[index, 'A'] + df.loc[index, 'B'] + df.loc[index, 'C'])

for_loop_df['sum'] = my_sums

end = time.time()

print(f'Time taken is {end-start} seconds')

for_loop_df.head()

Time taken is 5.149519920349121 seconds


,A,B,C,sum
0,65,29,34,128
1,72,60,52,184
2,63,97,24,184
3,95,47,27,169
4,37,96,23,156


**Iterrows**

In [156]:
iterrows_df = df.copy()

start = time.time()

my_sums = []

for index, row in df.iterrows():
    my_sums.append(row['A'] + row['B'] + row['C'])

iterrows_df['sum'] = my_sums

end = time.time()

print(f'Time taken is {end-start} seconds')

iterrows_df.head()

Time taken is 5.68018102645874 seconds


,A,B,C,sum
0,65,29,34,128
1,72,60,52,184
2,63,97,24,184
3,95,47,27,169
4,37,96,23,156


**Apply**

In [189]:
apply_df = df.copy()

def sum_apply(row):
    return row['A'] + row['B'] + row['C']

start = time.time()

apply_df['sums'] = df.apply(sum_apply, axis=1)

end = time.time()

print(f'Time taken is {end-start} seconds')

apply_df.head()


Time taken is 1.872981309890747 seconds


,A,B,C,sums
0,65,29,34,128
1,72,60,52,184
2,63,97,24,184
3,95,47,27,169
4,37,96,23,156


**Vectorized**

In [196]:
vectorized_df = df.copy()

start = time.time()

vectorized_df['sum'] = df['A'] + df['B'] + df['C']

end = time.time()

print(f'Time taken is {end-start} seconds')

vectorized_df.head()

Time taken is 0.002852201461791992 seconds


,A,B,C,sum
0,65,29,34,128
1,72,60,52,184
2,63,97,24,184
3,95,47,27,169
4,37,96,23,156


> **Reflektion:** Hur mycket snabbare var den vektoriserade varianten jämfört med t.ex. `for`‑loop?
>
> Kom ihåg:
> - `for` + `loc` → **sämst**
> - `iterrows()` → fortfarande långsamt, också **sämst**
> - `apply(axis=1)` → bättre, men inte bra
> - **vektoriserat** → nästan alltid bäst


---
## 4. Affärsexempel: timtaxa för el

Anta att vi har timvärden för elanvändning (`energy_kwh`) och vill räkna ut kostnaden per rad beroende på timme på dygnet:

- **Peak (17–24)**: 0.28 kr/kWh  
- **Off‑peak (0–7)**: 0.18 kr/kWh  
- **Övrig tid**: 0.22 kr/kWh  

In [198]:
demand_profile_df.head()

,date_time,energy_kwh
0,2013-01-01 00:00:00,0.586
1,2013-01-01 01:00:00,0.580
2,2013-01-01 02:00:00,0.572
3,2013-01-01 03:00:00,0.596
4,2013-01-01 04:00:00,0.592


In [234]:
def calculate_cost_apply(row):

    hour = row['date_time'].hour

    if 17 <= hour <= 24:
        rate = 0.28
    elif 0 <= hour < 7:
        rate = 0.18
    else:
        rate = 0.22

    return row['energy_kwh'] * rate


start = time.time()
demand_profile_df['cost_apply'] = demand_profile_df.apply(calculate_cost_apply, axis=1)
end = time.time()

print(f'Time taken is {end-start} seconds')

demand_profile_df.head()

Time taken is 0.03209376335144043 seconds


,date_time,energy_kwh,cost_apply
0,2013-01-01 00:00:00,0.586,0.10548
1,2013-01-01 01:00:00,0.580,0.10440
2,2013-01-01 02:00:00,0.572,0.10296
3,2013-01-01 03:00:00,0.596,0.10728
4,2013-01-01 04:00:00,0.592,0.10656


### 4.1 Vektoriserad version med boolean‑masker och `np.select`

Nu gör vi samma sak utan `apply`, helt vektoriserat.


In [235]:
demand_profile_df.head()

,date_time,energy_kwh,cost_apply
0,2013-01-01 00:00:00,0.586,0.10548
1,2013-01-01 01:00:00,0.580,0.10440
2,2013-01-01 02:00:00,0.572,0.10296
3,2013-01-01 03:00:00,0.596,0.10728
4,2013-01-01 04:00:00,0.592,0.10656


In [ ]:
peak_mask = demand_profile_df['date_time'].dt.hour.between(17, 24)
off_peak_mask = demand_profile_df['date_time'].dt.hour.between(0, 7)

conditions = [peak_mask, off_peak_mask]
rates = [0.28, 0.18]
default_rate = 0.22

start = time.time()


demand_profile_df['rate_vec'] = np.select(conditions, rates, default=default_rate)
demand_profile_df['cost_vectorized'] = demand_profile_df['rate_vec'] * demand_profile_df['energy_kwh']

end = time.time()

print(f'Time taken is {end-start} seconds')


demand_profile_df.head(10)

Time taken is 0.00039696693420410156 seconds


,date_time,energy_kwh,cost_apply,rate_vec,cost_vectorized
0,2013-01-01 00:00:00,0.586,0.10548,0.18,0.10548
1,2013-01-01 01:00:00,0.580,0.10440,0.18,0.10440
2,2013-01-01 02:00:00,0.572,0.10296,0.18,0.10296
3,2013-01-01 03:00:00,0.596,0.10728,0.18,0.10728
4,2013-01-01 04:00:00,0.592,0.10656,0.18,0.10656
5,2013-01-01 05:00:00,0.592,0.10656,0.18,0.10656
6,2013-01-01 06:00:00,0.596,0.10728,0.18,0.10728
7,2013-01-01 07:00:00,0.239,0.05258,0.18,0.04302
8,2013-01-01 08:00:00,0.566,0.12452,0.22,0.12452
9,2013-01-01 09:00:00,0.557,0.12254,0.22,0.12254


---
## 5. Minne och datatyper (dtypes) – en snabb titt

Prestanda handlar inte bara om tid utan också om **minne**.

- Mindre datatyper → mindre minne → kan ofta ge bättre prestanda.
- Exempel: `int64` → `int32` eller `int16` om värdeintervallet är litet.

Låt oss titta på minnesanvändningen för vårt `demand_profile_df`.


In [261]:
# byte till megabyte

print(f'En Megabyte är precis {1024**2} bytes')

En Megabyte är precis 1048576 bytes


In [246]:
demand_profile_df.dtypes

date_time          datetime64[ns]
energy_kwh                float64
cost_apply                float64
rate_vec                  float64
cost_vectorized           float64
dtype: object

In [245]:
# Minnesanvändning i bytes

demand_profile_df.memory_usage(deep=True)

Index                132
date_time          70080
energy_kwh         70080
cost_apply         70080
rate_vec           70080
cost_vectorized    70080
dtype: int64

In [248]:
total_memory_mb = demand_profile_df.memory_usage(deep=True).sum() / (1024**2)

print(f'Total minnesanvändning: {total_memory_mb} MB')

Total minnesanvändning: 0.3342933654785156 MB


### 5.1 Enkel nedkastning (downcasting) av numeriska kolumner

Vi kan prova att nedkasta numeriska kolumner och se om minnet minskar.


In [253]:
numerical_columns = demand_profile_df.columns[1:]

numerical_columns

Index(['energy_kwh', 'cost_apply', 'rate_vec', 'cost_vectorized'], dtype='object')

In [258]:
demand_profile_small_df = demand_profile_df.copy()

for column in numerical_columns:
    demand_profile_small_df[column] = pd.to_numeric(demand_profile_small_df[column], downcast='float')

demand_profile_small_df.info()



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8760 entries, 0 to 8759
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   date_time        8760 non-null   datetime64[ns]
 1   energy_kwh       8760 non-null   float32       
 2   cost_apply       8760 non-null   float32       
 3   rate_vec         8760 non-null   float32       
 4   cost_vectorized  8760 non-null   float32       
dtypes: datetime64[ns](1), float32(4)
memory usage: 205.4 KB


In [259]:
total_memory_mb = demand_profile_small_df.memory_usage(deep=True).sum() / (1024**2)

print(f'Total minnesanvändning: {total_memory_mb} MB')

Total minnesanvändning: 0.20062637329101562 MB


> **Notera:** I verkliga projekt vill man vara försiktig med precision när man nedkastar.
>
> Poängen här är att **dtypes spelar roll** för hur mycket minne en DataFrame tar,
> vilket i sin tur kan påverka prestanda.


---
## 6. Checklista för snabbare Pandas‑kod

När du skriver Pandas‑kod, tänk:

1. **Kan jag använda en inbyggd Pandas/NumPy‑funktion?**  
   – t.ex. `.str.*`, `.dt.*`, `.where`, `.clip`, `groupby().agg(...)`

2. **Kan jag formulera problemet vektoriserat?**  
   – med boolean‑masker, `np.select`, `np.where`, etc.

3. **Undvik Python‑loopar över rader** om det går:  
   – undvik `for` + `loc`, `iterrows()`, och använd `apply(axis=1)` endast som sista utväg.

4. **Tänk på dtypes och minne**:  
   – onödigt stora datatyper → mer minne → kan bli långsammare.

5. **Mät innan du optimerar för mycket**:  
   – använd `%timeit` eller enkla `time.time()`‑mätningar för att se vad som faktiskt är långsamt.


---
## 7. Extra läsning

Om du vill fördjupa dig:

- *Fast, Flexible, and Easy Data Analysis with Pandas* (Real Python)  
- Pandas officiella dokumentation om [Optimization & Performance](https://pandas.pydata.org/docs/user_guide/enhancingperf.html)

---

📝 **Förslag på egen övning:**  
Välj en bit kod från ett tidigare labb/uppgift, och:
1. Mät hur lång tid den tar.  
2. Försök skriva om den mer vektoriserat.  
3. Mät igen och jämför.
